## Clustering Analysis - FIFA 23 Players

Σε αυτό το notebook εφαρμόζεται ομαδοποίηση (clustering) σε δεδομένα παικτών από το FIFA 23.

Ο στόχος είναι να βρούμε ομάδες παικτών με παρόμοια χαρακτηριστικά, όπως overall rating, pace, shooting, passing, dribbling, defending και physicality.

Σε αντίθεση με το regression και το classification, στο clustering δεν υπάρχει target variable. Το μοντέλο δεν προσπαθεί να προβλέψει μια γνωστή απάντηση, αλλά προσπαθεί να ανακαλύψει φυσικές ομάδες μέσα στα δεδομένα.

## 1. import Libraries

In [1]:
# Βασικές βιβλιοθήκες για επεξεργασία δεδομένων
import pandas as pd
import numpy as np

# Βιβλιοθήκη για γραφήματα
import matplotlib.pyplot as plt

# Εργαλεία για διαχωρισμό και επεξεργασία δεδομένων
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Αλγόριθμοι clustering
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN

# Metrics για αξιολόγηση clustering
from sklearn.metrics import silhouette_score

# PCA για απεικόνιση των clusters σε 2 διαστάσεις
from sklearn.decomposition import PCA

# Για να εμφανίζονται τα γραφήματα μέσα στο notebook
%matplotlib inline

## 2. Loading and Description of Dataset

Σε αυτό το βήμα φορτώνουμε το FIFA 23 players dataset και ελέγχουμε τη βασική του δομή.

Το dataset περιέχει πληροφορίες για ποδοσφαιριστές, όπως ηλικία, ύψος, βάρος, overall rating, potential και τεχνικά χαρακτηριστικά.

In [2]:
# Φόρτωση του dataset
df = pd.read_csv('data/fifa23_players_clean.csv')
df.head() # Βλέπουμε τις 5 πρώτες γραμμές του dataset

,id,name,full_name,age,height,weight,photo_url,nationality,overall,potential,...,lm_rating,cm_rating,rm_rating,lwb_rating,cdm_rating,rwb_rating,lb_rating,cb_rating,rb_rating,gk_rating
0,158023,L. Messi,Lionel Messi,35,169,67,https://cdn.sofifa.net/players/158/023/23_60.png,Argentina,91,91,...,91,87,91,67,65,67,62,53,62,22
1,165153,K. Benzema,Karim Benzema,34,187,81,https://cdn.sofifa.net/players/165/153/23_60.png,France,91,91,...,89,84,89,67,67,67,63,58,63,21
2,188545,R. Lewandowski,Robert Lewandowski,33,185,81,https://cdn.sofifa.net/players/188/545/23_60.png,Poland,91,91,...,86,83,86,67,69,67,64,63,64,22
3,190871,Neymar Jr,Neymar da Silva Santos Jr.,30,175,68,https://cdn.sofifa.net/players/190/871/23_60.png,Brazil,91,91,...,91,85,91,69,66,69,65,54,65,23
4,192119,T. Courtois,Thibaut Courtois,30,199,96,https://cdn.sofifa.net/players/192/119/23_60.png,Belgium,91,92,...,34,36,34,32,34,32,32,32,32,91


In [3]:
# Βασικές πληροφορίες στηλών και τύπων δεδομένων
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1221 entries, 0 to 1220
Data columns (total 82 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   1221 non-null   int64 
 1   name                 1221 non-null   object
 2   full_name            1221 non-null   object
 3   age                  1221 non-null   int64 
 4   height               1221 non-null   int64 
 5   weight               1221 non-null   int64 
 6   photo_url            1221 non-null   object
 7   nationality          1221 non-null   object
 8   overall              1221 non-null   int64 
 9   potential            1221 non-null   int64 
 10  growth               1221 non-null   int64 
 11  total_stats          1221 non-null   int64 
 12  base_stats           1221 non-null   int64 
 13  positions            1221 non-null   object
 14  best_position        1221 non-null   object
 15  club                 1221 non-null   object
 16  nation

In [4]:
#Ονόματα στηλών
df.columns 

Index(['id', 'name', 'full_name', 'age', 'height', 'weight', 'photo_url',
       'nationality', 'overall', 'potential', 'growth', 'total_stats',
       'base_stats', 'positions', 'best_position', 'club', 'national_team',
       'national_position', 'national_number', 'preferred_foot',
       'int_reputation', 'weak_foot', 'skill_moves', 'attacking_work_rate',
       'defensive_work_rate', 'pace_total', 'shooting_total', 'passing_total',
       'dribbling_total', 'defending_total', 'physicality_total', 'crossing',
       'finishing', 'heading_accuracy', 'short_passing', 'volleys',
       'dribbling', 'curve', 'fk_accuracy', 'long_passing', 'ball_control',
       'acceleration', 'sprint_speed', 'agility', 'reactions', 'balance',
       'shot_power', 'jumping', 'stamina', 'strength', 'long_shots',
       'aggression', 'interceptions', 'positioning', 'vision', 'penalties',
       'composure', 'marking', 'standing_tackle', 'sliding_tackle',
       'gk_diving', 'gk_handling', 'gk_kicking', '

In [5]:
print("Μέγεθος dataset:", df.shape)

Μέγεθος dataset: (1221, 82)
